# Slope vs System Size From Extracted Plot Legends

The slope values below were transcribed from the legends in the four supplied plots. The system sizes are `Nx=20` with `Ny = 30, 40, 50, 60`, ordered from left to right / top to bottom as specified.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Extracted Slopes

In [ ]:
slope_rows = [
    # Ny, config, left-panel slope, right-panel slope
    {"Nx": 20, "Ny": 30, "config": "overlap", "left_panel_slope": 0.5589, "right_panel_slope": 0.4761},
    {"Nx": 20, "Ny": 30, "config": "truncated", "left_panel_slope": 0.5104, "right_panel_slope": 0.4272},
    {"Nx": 20, "Ny": 40, "config": "overlap", "left_panel_slope": 0.4894, "right_panel_slope": 0.4436},
    {"Nx": 20, "Ny": 40, "config": "truncated", "left_panel_slope": 0.4541, "right_panel_slope": 0.3993},
    {"Nx": 20, "Ny": 50, "config": "overlap", "left_panel_slope": 0.4591, "right_panel_slope": 0.4282},
    {"Nx": 20, "Ny": 50, "config": "truncated", "left_panel_slope": 0.4285, "right_panel_slope": 0.3942},
    {"Nx": 20, "Ny": 60, "config": "overlap", "left_panel_slope": 0.4413, "right_panel_slope": 0.4174},
    {"Nx": 20, "Ny": 60, "config": "truncated", "left_panel_slope": 0.4071, "right_panel_slope": 0.3804},
]

slopes = pd.DataFrame(slope_rows)
slopes

## Slope vs System Size

In [ ]:
colors = {"overlap": "tab:red", "truncated": "tab:blue"}
markers = {"overlap": "o", "truncated": "s"}

plot_specs = [
    ("left_panel_slope", "Entanglement Scaling | Fitted Slope"),
    ("right_panel_slope", "I2-I3 Scaling | Fitted Slope"),
]

fig, axes = plt.subplots(
    3,
    2,
    figsize=(10.5, 11.0),
    sharex=False,
)

fig.subplots_adjust(
    left=0.08,
    right=0.98,
    bottom=0.07,
    top=0.94,
    wspace=0.32,
    hspace=0.55,
)

for col_idx, (slope_col, title) in enumerate(plot_specs):
    ax_slope = axes[0, col_idx]
    ax_log_slope = axes[1, col_idx]
    ax_loglog = axes[2, col_idx]

    for config, group in slopes.groupby("config", sort=False):
        group = group.sort_values("Ny")
        ny = group["Ny"].to_numpy(dtype=float)
        slope = group[slope_col].to_numpy(dtype=float)

        valid = np.isfinite(ny) & np.isfinite(slope) & (ny > 0) & (slope > 0)
        ny = ny[valid]
        slope = slope[valid]

        ax_slope.plot(
            ny,
            slope,
            marker=markers[config],
            color=colors[config],
            lw=1.8,
            ms=5,
            label=config,
        )
        ax_log_slope.plot(
            ny,
            np.log(slope),
            marker=markers[config],
            color=colors[config],
            lw=1.8,
            ms=5,
            label=config,
        )
        ax_loglog.plot(
            np.log(ny),
            np.log(slope),
            marker=markers[config],
            color=colors[config],
            lw=1.8,
            ms=5,
            label=config,
        )

        for x, y in zip(ny, slope):
            ax_slope.annotate(
                f"{y:.4f}",
                (x, y),
                textcoords="offset points",
                xytext=(0, 6),
                ha="center",
                fontsize=8,
                color=colors[config],
            )

    ax_slope.set_title(title)
    ax_slope.set_xlabel("Ny, with Nx=20")
    ax_slope.set_ylabel("extracted slope")
    ax_slope.set_xticks(sorted(slopes["Ny"].unique()))
    ax_slope.legend(frameon=True, facecolor="white", framealpha=1.0, edgecolor="0.35")

    ax_log_slope.set_title(f"{title} | log slope vs Ny")
    ax_log_slope.set_xlabel("Ny, with Nx=20")
    ax_log_slope.set_ylabel("log(extracted slope)")
    ax_log_slope.set_xticks(sorted(slopes["Ny"].unique()))
    ax_log_slope.legend(frameon=True, facecolor="white", framealpha=1.0, edgecolor="0.35")

    ax_loglog.set_title(f"{title} | log slope vs log Ny")
    ax_loglog.set_xlabel("log(Ny)")
    ax_loglog.set_ylabel("log(extracted slope)")
    ax_loglog.legend(frameon=True, facecolor="white", framealpha=1.0, edgecolor="0.35")

plt.show()


## Truncation-Overlap Difference

In [ ]:
wide = slopes.pivot(index="Ny", columns="config", values=["left_panel_slope", "right_panel_slope"])
delta = pd.DataFrame({
    "Nx": 20,
    "Ny": wide.index,
    "left_panel_overlap_minus_truncated": wide[("left_panel_slope", "overlap")] - wide[("left_panel_slope", "truncated")],
    "right_panel_overlap_minus_truncated": wide[("right_panel_slope", "overlap")] - wide[("right_panel_slope", "truncated")],
}).reset_index(drop=True)
delta

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5.2, 3.6), constrained_layout=True)
ax.plot(delta["Ny"], delta["left_panel_overlap_minus_truncated"], marker="o", lw=1.8, label="left panel")
ax.plot(delta["Ny"], delta["right_panel_overlap_minus_truncated"], marker="s", lw=1.8, label="right panel")
ax.set_xlabel("Ny, with Nx=20")
ax.set_ylabel("slope(overlap) - slope(truncated)")
ax.set_xticks(sorted(delta["Ny"].unique()))
ax.legend(frameon=True, facecolor="white", framealpha=1.0, edgecolor="0.35")
ax.set_title("Slope separation vs system size")
plt.show()